<a href="https://colab.research.google.com/github/karimboyevshaxram97-ux/computer_vision/blob/main/menu_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print('Menu Detector')

Menu Detector


In [3]:
# -----------------------------
# Import Libraries
# -----------------------------

from google.colab import drive

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader

import os
import numpy as np


In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Define Dataset Path
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",  # label grouping | class consolidation
    "cheesecake": "dessert",      # label grouping | class consolidation
    "kebab": "kebab",
    "pilaf": "pilaf"
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)


transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Tasvirni 224x224 pikselga o‘lchamini o‘zgartiradi
    transforms.ToTensor(),           # Tasvirni PyTorch tensoriga aylantiradi (modelga kiritish uchun)
    transforms.Normalize(            # Tasvir piksel qiymatlarini normallashtiradi
        mean=[0.485, 0.456, 0.406],  # RGB kanallari uchun o‘rtacha qiymatlar
        std=[0.229, 0.224, 0.225]    # RGB kanallari uchun standart og‘ishlar
    )
])






Dataset_path: /content/drive/MyDrive/food101_dataset
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [6]:
# -------------------------
# Custom Dataset Class
# -------------------------
class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images              # Tasvirlar ro‘yxati (fayl yo‘llari)
        self.labels = labels              # Har bir tasvirga mos keladigan label (klass)
        self.transform = transform        # Tasvirlarga qo‘llanadigan transformatsiyalar

    def __len__(self):
        print('images_length', len(self.images))  # Dataset uzunligini konsolga chiqaradi
        return len(self.images)                   # Datasetdagi tasvirlar sonini qaytaradi

    def __getitem__(self, idx):
        img_path = self.images[idx]               # Index bo‘yicha tasvir yo‘lini oladi
        print('image_path', img_path)             # Tasvir yo‘lini konsolga chiqaradi
        label = self.labels[idx]                  # Index bo‘yicha labelni oladi
        print('label', label)                     # Labelni konsolga chiqaradi
        try:
            image = Image.open(img_path).convert('RGB')  # Tasvirni ochadi va RGB formatga o‘tkazadi
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")  # Agar tasvir buzilgan bo‘lsa, xabar chiqaradi
            return self.__getitem__((idx + 1) % len(self.images))  # Keyingi tasvirni qaytaradi
        if self.transform:
            image = self.transform(image)         # Agar transform berilgan bo‘lsa, tasvirga qo‘llaydi
        return image, label                       # Tasvir va labelni qaytaradi


In [7]:
# ------------------------------
# Gather and Split Data
# ------------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)   # Klass papkasining yo‘lini hosil qiladi
    print('class_path:', class_path)
    if not os.path.exists(class_path):                        # Agar papka mavjud bo‘lmasa, ogohlantiradi
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):                        # Papkadagi barcha fayllarni ko‘rib chiqadi
        if img.endswith(('.jpg', '.jpeg', '.png')):           # Faqat rasm fayllarini tanlaydi
            full_path = os.path.join(class_path, img)         # Faylning to‘liq yo‘lini hosil qiladi
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))  # Fayl yo‘li va label indeksini qo‘shadi

np.random.shuffle(all_images)                                # Tasvirlarni aralashtiradi (random)
split = int(0.8 * len(all_images))                           # 80% trening, 20% validatsiya uchun ajratadi
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)                # Trening tasvirlar va label’larni ajratadi
val_images, val_labels = zip(*val_data)                      # Validatsiya tasvirlar va label’larni ajratadi

print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)            # Custom dataset obyektini yaratadi
print(len(dataset))                                          # Dataset uzunligini chiqaradi
img, lbl = dataset[0]                                        # Birinchi tasvir va labelni oladi


class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
all_images: [('/content/drive/MyDrive/food101_dataset/kebab/Image_36.jpg', 3), ('/content/drive/MyDrive/food101_dataset/kebab/Image_12.jpg', 3), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_27.jpg', 4), ('/content/drive/MyDrive/food101_dataset/kebab/Image_19.jpg', 3), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_25.jpeg', 4), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_28.jpg', 4), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_21.jpg', 4), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_49.jpg', 4), ('/content/drive/MyDrive/food101_dataset/pilaf/Image_51.jpg', 4), ('/content/drive/MyDrive/food1

In [8]:
train_dataset = FoodDataset(
    train_images,          # Trening uchun tasvirlar ro‘yxati
    train_labels,          # Trening uchun label’lar ro‘yxati
    transform=transform    # Tasvirlarga qo‘llanadigan transformatsiyalar (resize, normalize, tensor)
)

val_dataset = FoodDataset(
    val_images,            # Validatsiya uchun tasvirlar ro‘yxati
    val_labels,            # Validatsiya uchun label’lar ro‘yxati
    transform=transform    # Tasvirlarga qo‘llanadigan transformatsiyalar
)


In [9]:
train_loader = DataLoader(
    train_dataset,   # Trening dataset (tasvirlar + label’lar)
    batch_size=32,   # Har bir batchda 32 ta tasvir bo‘ladi
    shuffle=True,    # Tasvirlarni aralashtirib beradi (model o‘rganishda bias bo‘lmasligi uchun)
    num_workers=2    # Ma’lumotlarni yuklash uchun 2 ta parallel ishchi jarayon
)

val_loader = DataLoader(
    val_dataset,     # Validatsiya dataset (tasvirlar + label’lar)
    batch_size=32,   # Har bir batchda 32 ta tasvir bo‘ladi
    shuffle=False,   # Validatsiyada tartibni saqlaydi (aralashtirmaydi)
    num_workers=2    # Ma’lumotlarni yuklash uchun 2 ta parallel ishchi jarayon
)


images_length 84
images_length 84


In [10]:
# ======================
# Pretrained Model
# ======================
model = mobilenet_v2(weights="IMAGENET1K_V1")   # MobilenetV2 modelini ImageNet datasetda oldindan o‘qitilgan (pretrained) vaznlar bilan chaqiradi

# ======================
# Fine-tuning uchun oxirgi layerni almashtirish
# ======================
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,  # Oxirgi fully-connected layer kirish o‘lchami
    NUM_CLASSES                       # Bizning datasetdagi klasslar soni (masalan: hamburger, kebab, pilaf va h.k.)
)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 216MB/s]


In [11]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'   # Agar GPU (CUDA) mavjud bo‘lsa, uni ishlatadi, aks holda CPU
)

print('device', device)                              # Qaysi qurilma tanlanganini konsolga chiqaradi

model = model.to(device)                             # Modelni tanlangan qurilmaga (GPU yoki CPU) ko‘chiradi


device cpu


In [12]:
criterion = nn.CrossEntropyLoss()   # Klassifikatsiya uchun loss function (cross-entropy).
                                    # Model chiqishi va haqiqiy label o‘rtasidagi xatolikni hisoblaydi.

optimizer = optim.Adam(
    model.parameters(),             # Model parametrlarini optimallashtiradi
    lr=0.001                        # O‘rganish tezligi (learning rate)
)

torch.backends.cudnn.benchmark = True   # GPU’da convolution operatsiyalarini tezlashtirish uchun
                                        # optimal konfiguratsiyani avtomatik tanlaydi
